# CALM-R3 — Calibration-Aware Reflective Routing & Reasoning
Standalone, runnable on Kaggle (2×T4). Loads your trained 7B LoRA adapter, then:
- **Phase 0** (auto, no training): validates whether reasoning beats the fast pass on uncertain items → prints **GO / NO-GO**.
- **Phase 1** (only if GO; set `RUN_PHASE1=True`): two-adapter confidence routing + corrected reflective loss.

Set `ADAPTER_SRC` in the CONFIG cell (or add the adapter as a Kaggle Input and leave it `"AUTO"`).
Artifacts → `paper_artifacts_v3_qwen7b/calm_r3/` and `calm_r3_artifacts.zip`.


In [ ]:
# === Secrets (optional, NO hardcoded keys) ===
import os
def _load_secret(name):
    v = os.environ.get(name)
    if v: return v
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: os.environ[name] = v; return v
    except Exception:
        pass
    return None
HF_TOKEN = _load_secret("HF_TOKEN")
print("HF_TOKEN:", bool(HF_TOKEN), "(optional)")

# === Installs ===
import sys, subprocess
def pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
def ensure(mod, pkg=None):
    try: __import__(mod)
    except Exception: pip(pkg or mod)
for mod, pkg in [("datasets","datasets"),("transformers","transformers>=4.41"),
                 ("accelerate","accelerate"),("sklearn","scikit-learn"),
                 ("matplotlib","matplotlib"),("bitsandbytes","bitsandbytes"),
                 ("peft","peft"),("huggingface_hub","huggingface_hub")]:
    ensure(mod,pkg)
print("installs ok")



In [ ]:
# === Imports & devices ===
import os, gc, json, csv, math, random, re, time, glob, urllib.request, warnings
from dataclasses import dataclass
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import numpy as np, torch
import torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, Dataset
assert torch.cuda.is_available(), "CUDA required."
N_GPU = torch.cuda.device_count()
STUDENT_DEVICE = torch.device("cuda:0")
OPTIONS = list("ABCD")
print("GPUs=", N_GPU, "|", torch.cuda.get_device_name(0))



In [ ]:
# ============================= CONFIG =============================
# Point ADAPTER_SRC at your trained 7B LoRA adapter. Three ways (auto-detected):
#   1) leave "AUTO" and add the adapter folder as a Kaggle Input (it finds adapter_config.json)
#   2) a HF Hub id, e.g. "sabarna07/your-medqa-7b-adapter"
#   3) a local path, e.g. "/kaggle/input/medqa-7b-adapter/best_student_adapter"
ADAPTER_SRC  = "AUTO"
BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ATTACH_EXISTING = True   # if a `student` already lives in this kernel, reuse it

# ---- CALM-R3 run knobs ----
REASON_TEST_N = 600      # test items to reason over (set 1273 for the full split)
RUN_PHASE1    = False    # set True ONLY after Phase 0 prints GO

@dataclass
class CFG:
    seed: int = 2000
    val_n: int = 1000
    test_n: int = 1273
    seed_labeled_n: int = 2000
    unlabeled_n: int = 4000
    eval_bs: int = 8
    max_src_len: int = 448
    ece_bins: int = 10
    reason_cot_tokens: int = 256
    grad_clip: float = 1.0
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_targets: tuple = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
    dataset_candidates: tuple = ("sabarna07/MedQA-USMLE-4-options-hf","GBaker/MedQA-USMLE-4-options-hf")
    local_data_dir: str = ""
    time_budget_hours: float = 11.0
    out_dir: str = "./paper_artifacts_v3_qwen7b"
cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
START_TIME = time.time(); DEADLINE = START_TIME + cfg.time_budget_hours*3600
def time_left(): return DEADLINE - time.time()
def have_time(need=0.0): return time_left() > need
print("config ready | out_dir:", cfg.out_dir)



In [ ]:
# ============================= DATA (same loader & splits as the main run) =============================
def _get_options(ex):
    o = ex.get("options") or ex.get("choices") or ex.get("endings")
    if isinstance(o, dict):
        return [str(o.get(k, o.get(k.lower(), ""))).strip() for k in OPTIONS]
    if isinstance(o, list) and len(o) >= 4:
        if isinstance(o[0], dict) and "value" in o[0]:
            o = [d["value"] for d in sorted(o, key=lambda x: x.get("key",""))]
        return [str(x).strip() for x in o[:4]]
    for ks in [("opa","opb","opc","opd"), ("ending0","ending1","ending2","ending3")]:
        if all(k in ex for k in ks): return [str(ex[k]).strip() for k in ks]
    return None
def normalize_example(ex):
    q = ex.get("sent1") or ex.get("question") or ex.get("stem") or ex.get("prompt") or ""
    if ex.get("sent2") and str(ex["sent2"]).strip() not in ("", "?"):
        q = (str(q).strip()+"\n"+str(ex["sent2"]).strip()).strip()
    opts = _get_options(ex) or ["","","",""]
    lab = ex.get("answer_idx", ex.get("label", ex.get("answer", None)))
    ans = None
    if isinstance(lab, int): ans = OPTIONS[lab] if 0<=lab<4 else None
    elif lab is not None and str(lab).strip().isdigit(): ans = OPTIONS[int(str(lab).strip())]
    elif isinstance(lab, str) and lab.strip().upper() in OPTIONS: ans = lab.strip().upper()
    elif isinstance(lab, str):
        for i,o in enumerate(opts):
            if o and o.strip().lower() == lab.strip().lower(): ans = OPTIONS[i]; break
    if ans is None: ans = "A"
    return {"question": str(q).strip(), "options": opts, "answer": ans}
def normalize_split(split, max_n=None):
    n = len(split) if max_n is None else min(len(split), max_n)
    return Dataset.from_list([normalize_example(split[i]) for i in range(n)])

def _read_records(text):
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list): return obj
        if isinstance(obj, dict):
            for v in obj.values():
                if isinstance(v, list): return v
            return [obj]
    except Exception: pass
    rows=[]
    for line in text.splitlines():
        line=line.strip()
        if not line: continue
        try: rows.append(json.loads(line))
        except Exception: pass
    return rows
def _http_get(url, token=None, timeout=180):
    h={"User-Agent":"Mozilla/5.0"}
    if token: h["Authorization"]="Bearer "+token
    with urllib.request.urlopen(urllib.request.Request(url, headers=h), timeout=timeout) as r:
        return r.read().decode("utf-8")
HF_DIRECT = [("sabarna07/MedQA-USMLE-4-options-hf", {"train":"train.json","validation":"dev.json","test":"test.json"}),
             ("GBaker/MedQA-USMLE-4-options-hf", {"train":"train.json","validation":"dev.json","test":"test.json"})]
raw=None; used_id=None
for repo, fnames in HF_DIRECT:
    try:
        base="https://huggingface.co/datasets/%s/resolve/main/"%repo; d={}
        for split, fn in fnames.items():
            recs=_read_records(_http_get(base+fn, token=HF_TOKEN))
            if not recs: raise ValueError("empty "+fn)
            d[split]=Dataset.from_list(recs)
        raw=d; used_id=repo+" (direct http)"
        print("Loaded:", repo, {k:len(v) for k,v in d.items()}); break
    except Exception as e:
        print("[http skip]", repo, type(e).__name__, str(e)[:80])
if raw is None:
    for cand in cfg.dataset_candidates:
        try:
            raw=load_dataset(cand, token=HF_TOKEN); used_id=cand
            print("Loaded via load_dataset:", cand, list(raw.keys())); break
        except Exception as e: print("[resolver skip]", cand, type(e).__name__)
assert raw is not None, "Dataset load failed: add the MedQA dataset as a Kaggle Input or check network."
sp=dict(raw)
if "validation" not in sp:
    tt=sp["train"].train_test_split(test_size=0.1, seed=cfg.seed); sp["train"],sp["validation"]=tt["train"],tt["test"]
if "test" not in sp: sp["test"]=sp["validation"]
full_train=normalize_split(sp["train"])
val_ds  = normalize_split(sp["validation"], max_n=cfg.val_n)
test_ds = normalize_split(sp["test"], max_n=cfg.test_n)
idx=list(range(len(full_train))); random.Random(cfg.seed).shuffle(idx)
seed_ds=Dataset.from_list([full_train[i] for i in idx[:cfg.seed_labeled_n]])
print("seed=%d val=%d test=%d"%(len(seed_ds),len(val_ds),len(test_ds)))



In [ ]:
# ============================= MODEL + ADAPTER + HELPERS =============================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

def _resolve_adapter(src):
    # 1) explicit existing local directory
    if src and src not in ("AUTO", "") and os.path.isdir(src):
        return src
    # 2) auto-search common Kaggle locations for a saved LoRA adapter
    roots = ["/kaggle/input", "/kaggle/working", cfg.out_dir, "."]
    hits = []
    for r in roots:
        hits += glob.glob(os.path.join(r, "**", "adapter_config.json"), recursive=True)
    hits = sorted(set(hits), key=len)
    if hits:
        found = [os.path.dirname(h) for h in hits]
        print("auto-found adapter(s):", found[:5])
        return found[0]
    # 3) otherwise treat a non-AUTO value as a HF Hub id (e.g. "user/medqa-7b-adapter")
    if src and src not in ("AUTO", ""):
        print("no local adapter found; trying as HF Hub id:", src)
        return src
    raise FileNotFoundError(
        "No adapter found. Searched %s and found no adapter_config.json.\n"
        "Fix ONE of these in the CONFIG cell:\n"
        "  (a) set ADAPTER_SRC to your HF Hub id, e.g. 'sabarna07/medqa-7b-adapter'; or\n"
        "  (b) on Kaggle click 'Add Input' -> add the dataset/notebook-output that holds\n"
        "      best_student_adapter (it will auto-mount under /kaggle/input); or\n"
        "  (c) run these cells in the SAME kernel as your training run so `student` exists."
        % roots)

if ATTACH_EXISTING and ("student" in globals()) and ("student_tok" in globals()):
    print("Reusing existing `student` from this kernel.")
else:
    ADAPTER = _resolve_adapter(ADAPTER_SRC); print("Loading adapter:", ADAPTER)
    student_tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, token=HF_TOKEN)
    student_tok.padding_side="left"; student_tok.truncation_side="left"
    if student_tok.pad_token is None: student_tok.pad_token = student_tok.eos_token
    _bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    _base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=_bnb,
                device_map={"": str(STUDENT_DEVICE)}, low_cpu_mem_usage=True, token=HF_TOKEN)
    _base.config.use_cache=False
    _base = prepare_model_for_kbit_training(_base, use_gradient_checkpointing=True)
    student = PeftModel.from_pretrained(_base, ADAPTER, is_trainable=True, token=HF_TOKEN)
    print("student ready:", BASE_MODEL, "+ LoRA")
student.eval()

def format_prompt(q, opts):
    return ("You are taking a USMLE-style multiple-choice exam.\n"
            "Choose the single best answer.\n\nQuestion: " + q + "\n\nOptions:\n"
            "A. " + opts[0] + "\nB. " + opts[1] + "\nC. " + opts[2] + "\nD. " + opts[3] + "\n\n"
            "Reply with ONLY one letter: A, B, C, or D.")
def stu_chat(user, system="You are a medical expert. Answer with a single letter."):
    if hasattr(student_tok, "apply_chat_template"):
        return student_tok.apply_chat_template(
            [{"role":"system","content":system},{"role":"user","content":user}],
            tokenize=False, add_generation_prompt=True)
    return system + "\n\n" + user
def _build_letter_ids(tok):
    out={}
    for L in OPTIONS:
        c=[]
        for v in [L, " "+L]:
            e=tok.encode(v, add_special_tokens=False)
            if len(e)==1: c.append(e[0])
        out[L]=c or [tok.encode(L, add_special_tokens=False)[0]]
    return out
STU_LETTER_IDS=_build_letter_ids(student_tok); print("letter ids:", STU_LETTER_IDS)
def _fwd_last(model, input_ids, attention_mask):
    for kw in ("logits_to_keep","num_logits_to_keep"):
        try: return model(input_ids=input_ids, attention_mask=attention_mask, **{kw:1}).logits[:, -1, :]
        except TypeError: continue
    return model(input_ids=input_ids, attention_mask=attention_mask).logits[:, -1, :]
def _letter_logits(last):
    cols=[torch.logsumexp(torch.stack([last[:, t] for t in STU_LETTER_IDS[L]], 0), 0) for L in OPTIONS]
    return torch.stack(cols, 1)
@torch.no_grad()
def student_option_logits(model, prompts, bs=None):
    bs=bs or cfg.eval_bs; model.eval(); out=[]
    for i in range(0, len(prompts), bs):
        chats=[stu_chat(p) for p in prompts[i:i+bs]]
        enc=student_tok(chats, return_tensors="pt", padding=True, truncation=True,
                        max_length=cfg.max_src_len).to(STUDENT_DEVICE)
        out.append(_letter_logits(_fwd_last(model, enc.input_ids, enc.attention_mask)).float().cpu()); del enc
    torch.cuda.empty_cache(); return torch.cat(out, 0)
def expected_calibration_error(correct, conf, n_bins=10):
    correct=np.asarray(correct,float); conf=np.asarray(conf,float)
    bins=np.linspace(0,1,n_bins+1); e=0.0; n=len(correct)
    for i in range(n_bins):
        lo,hi=bins[i],bins[i+1]
        m=(conf>lo)&(conf<=hi) if i>0 else (conf>=lo)&(conf<=hi)
        if m.sum()==0: continue
        e+=(m.sum()/n)*abs(correct[m].mean()-conf[m].mean())
    return float(e)
def fit_temperature(logits, labels, max_iter=200):
    logits=torch.tensor(np.asarray(logits),dtype=torch.float32); labels=torch.tensor(np.asarray(labels),dtype=torch.long)
    logT=torch.zeros(1, requires_grad=True); opt=torch.optim.LBFGS([logT], lr=0.1, max_iter=max_iter)
    def closure():
        opt.zero_grad(); loss=F.cross_entropy(logits/logT.exp(), labels); loss.backward(); return loss
    opt.step(closure); return float(logT.exp().item())
def _parse_letter(t):
    if not t: return None
    pats=[r"\\boxed\{\s*([ABCD])\b", r"final\s+answer[^A-Za-z]{0,20}([ABCD])\b",
          r"correct\s+(?:answer|option|choice)\s*(?:is|:|=)?\s*\(?\s*([ABCD])\b",
          r"\banswer\s*(?:is|:|=)\s*\(?\s*([ABCD])\b", r"(?:^|\n)\s*\(?([ABCD])\)?\s*[.)]?\s*$"]
    for p in pats:
        ms=list(re.finditer(p, t, re.I))
        if ms: return ms[-1].group(1).upper()
    return None
print("helpers ready")




## CALM-R3 helpers (reasoning path)


In [ ]:
R3_DIR = os.path.join(cfg.out_dir, "calm_r3"); os.makedirs(R3_DIR, exist_ok=True)

# ---- knobs (raise for the full run; defaults are budget-safe) ----
REASON_TEST_N   = 300          # test items to generate a rationale for (1273 = full split)
REASON_MAXTOK   = cfg.reason_cot_tokens
MIN_REASON_TOK  = 96   # force real chain-of-thought (the student short-circuits otherwise)   # 256 by default
ROUTE_GRID      = np.round(np.linspace(0.30, 0.95, 14), 3)  # fast-confidence thresholds tau
RUN_PHASE1      = False         # <-- set True ONLY after Phase 0 says GO


def _sm(z, T=1.0):
    z = np.asarray(z) / T; z = z - z.max(-1, keepdims=True)
    e = np.exp(z); return e / e.sum(-1, keepdims=True)

def _use_cache(flag):
    try: student.config.use_cache = flag
    except Exception: pass

# ---------- reasoning path (single deterministic chain, then read option logits) ----------
@torch.no_grad()
def gen_rationale(q, opts, max_new=REASON_MAXTOK):
    user = ("You are an attending physician. Reason about THIS case BEFORE answering. "
            "In 3-5 sentences, compare the options and state the key differentiating "
            "feature. Do NOT reveal the letter until the very end. "
            "Finish with a separate final line exactly: 'Final Answer: X'.\nQuestion: " + q +
            "\nA. " + opts[0] + "\nB. " + opts[1] + "\nC. " + opts[2] + "\nD. " + opts[3])
    enc = student_tok([stu_chat(user)], return_tensors="pt", truncation=True,
                      max_length=cfg.max_src_len).to(STUDENT_DEVICE)
    gen = student.generate(**enc, do_sample=False, min_new_tokens=MIN_REASON_TOK,
                           max_new_tokens=max_new, pad_token_id=student_tok.pad_token_id)
    new = gen[0, enc.input_ids.shape[1]:]
    txt = student_tok.decode(new, skip_special_tokens=True)
    ntok = int((new != student_tok.pad_token_id).sum().item())
    del enc, gen; torch.cuda.empty_cache()
    return txt, ntok, _parse_letter(txt)

@torch.no_grad()
def reason_logits(q, opts, rationale):
    su = (format_prompt(q, opts) + "\n\nStep-by-step reasoning:\n" + rationale.strip() +
          "\n\nGiven the reasoning above, reply with ONLY one letter: A, B, C, or D.")
    enc = student_tok([stu_chat(su)], return_tensors="pt", truncation=True,
                      max_length=cfg.max_src_len + REASON_MAXTOK + 64).to(STUDENT_DEVICE)
    last = _fwd_last(student, enc.input_ids, enc.attention_mask)
    out = _letter_logits(last)[0].float().cpu().numpy()
    del enc, last; torch.cuda.empty_cache()
    return out


# =====================================================================================



## PHASE 0 — routing diagnostic (the validation)


In [ ]:
# PHASE 0 -- ROUTING DIAGNOSTIC  (the validation)
# =====================================================================================
def phase0():
    import random
    print("\n" + "=" * 70 + "\nPHASE 0  CALM-R3 routing diagnostic (no training)\n" + "=" * 70)
    student.eval()

    # fast path: calibrate temperature on validation
    vlog = student_option_logits(student, [format_prompt(e["question"], e["options"]) for e in val_ds]).numpy()
    vgold = np.array([OPTIONS.index(e["answer"]) for e in val_ds])
    T = fit_temperature(vlog, vgold); print("fast-path temperature T=%.3f" % T)

    # choose a test subset
    rng = random.Random(cfg.seed); idx = list(range(len(test_ds))); rng.shuffle(idx)
    idx = idx[:min(REASON_TEST_N, len(test_ds))]
    sub = [test_ds[i] for i in idx]
    gold = np.array([OPTIONS.index(e["answer"]) for e in sub])

    # fast predictions + calibrated confidence
    flog = student_option_logits(student, [format_prompt(e["question"], e["options"]) for e in sub]).numpy()
    fp = _sm(flog, T); fast_pred = fp.argmax(1); fast_conf = fp.max(1)
    fast_correct = (fast_pred == gold).astype(int)

    # reasoning predictions (one chain each) + token cost
    _use_cache(True)
    r_pred = np.zeros(len(sub), int); r_tok = np.zeros(len(sub)); r_ok_parse = np.zeros(len(sub), int)
    t0 = time.time()
    for j, e in enumerate(sub):
        if not have_time(600):
            print("  [budget] stopping reasoning gen at", j);
            r_pred = r_pred[:j]; r_tok = r_tok[:j]; gold = gold[:j]
            fast_pred = fast_pred[:j]; fast_conf = fast_conf[:j]; fast_correct = fast_correct[:j]; break
        rat, ntok, letter = gen_rationale(e["question"], e["options"])
        if letter not in OPTIONS:   # fall back to logit read over the rationale
            letter = OPTIONS[int(reason_logits(e["question"], e["options"], rat).argmax())]
        r_pred[j] = OPTIONS.index(letter); r_tok[j] = ntok
        if (j + 1) % 50 == 0: print("  reasoned %d/%d  (%.1f min)" % (j + 1, len(sub), (time.time() - t0) / 60))
    _use_cache(False)
    n = len(gold); reason_correct = (r_pred == gold).astype(int)

    fast_acc = float(fast_correct.mean()); reason_acc = float(reason_correct.mean())
    mean_rtok = float(r_tok[:n].mean())
    print("\nsubset n=%d | fast acc=%.4f | reason acc=%.4f | mean reason tokens=%.1f"
          % (n, fast_acc, reason_acc, mean_rtok))

    # ---- per-decile diagnostic: does reasoning beat fast where fast is unsure? ----
    order = np.argsort(fast_conf); buckets = np.array_split(order, 10)
    rows = []
    for bi, b in enumerate(buckets):
        rows.append(dict(decile=bi + 1, n=len(b),
                         conf_lo=float(fast_conf[b].min()), conf_hi=float(fast_conf[b].max()),
                         fast_acc=float(fast_correct[b].mean()), reason_acc=float(reason_correct[b].mean()),
                         reason_minus_fast=float(reason_correct[b].mean() - fast_correct[b].mean())))
    try:
        with open(os.path.join(R3_DIR, "phase0_buckets.csv"), "w", newline="") as fcsv:
            w = csv.DictWriter(fcsv, fieldnames=list(rows[0].keys())); w.writeheader(); [w.writerow(r) for r in rows]
    except Exception as _e: print("  [warn] save phase0_buckets.csv:", _e)
    print("\n decile  conf-range      fast   reason   reason-fast")
    for r in rows:
        print("  %2d  [%.2f,%.2f]   %.3f   %.3f    %+.3f"
              % (r["decile"], r["conf_lo"], r["conf_hi"], r["fast_acc"], r["reason_acc"], r["reason_minus_fast"]))

    # ---- threshold sweep: route conf<tau to reasoning; accuracy vs mean tokens/q ----
    sweep = []
    for tau in ROUTE_GRID:
        route = fast_conf < tau
        pred = np.where(route, r_pred, fast_pred)
        acc = float((pred == gold).mean()); frac = float(route.mean())
        toks = float(np.where(route, r_tok[:n], 0.0).mean())   # fast path generates ~0 tokens
        sweep.append(dict(tau=float(tau), routed_acc=acc, frac_reasoned=frac, mean_tokens=toks,
                          ace=acc / (toks + 1.0)))
    try:
        with open(os.path.join(R3_DIR, "phase0_sweep.csv"), "w", newline="") as fcsv:
            w = csv.DictWriter(fcsv, fieldnames=list(sweep[0].keys())); w.writeheader(); [w.writerow(r) for r in sweep]
    except Exception as _e: print("  [warn] save phase0_sweep.csv:", _e)
    best = max(sweep, key=lambda r: r["routed_acc"])

    # ---- GO / NO-GO verdict ----
    low = np.concatenate([buckets[0], buckets[1], buckets[2]])   # bottom 30% confidence
    reason_helps_low = float(reason_correct[low].mean() - fast_correct[low].mean())
    go = (best["routed_acc"] > fast_acc + 0.005) and (reason_helps_low > 0)
    verdict = "GO" if go else "NO-GO"
    print("\n" + "-" * 70)
    print("best routed acc = %.4f @ tau=%.2f (%.0f%% reasoned, %.1f tok/q)"
          % (best["routed_acc"], best["tau"], 100 * best["frac_reasoned"], best["mean_tokens"]))
    print("fast-only acc   = %.4f   |   reason-only acc = %.4f" % (fast_acc, reason_acc))
    print("reason - fast on bottom-30%% confidence = %+.4f" % reason_helps_low)
    print("VERDICT: %s  -- %s" % (verdict,
          "routing helps; Phase 1 training is justified." if go else
          "reasoning does NOT beat fast where it matters; do NOT train Phase 1 for this student."))
    print("-" * 70)

    # ---- plots ----
    plt.figure(figsize=(5, 3.4))
    xs = [r["decile"] for r in rows]
    plt.plot(xs, [r["fast_acc"] for r in rows], "o-", label="fast")
    plt.plot(xs, [r["reason_acc"] for r in rows], "s-", label="reason")
    plt.xlabel("fast-confidence decile (1=least sure)"); plt.ylabel("accuracy")
    plt.title("Fast vs reasoning by confidence"); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(R3_DIR, "phase0_bucket_acc.png"), dpi=200); plt.close()

    plt.figure(figsize=(5, 3.4))
    plt.plot([r["mean_tokens"] for r in sweep], [r["routed_acc"] for r in sweep], "o-")
    plt.scatter([0], [fast_acc], c="k", marker="x", s=60, label="always-fast")
    plt.scatter([mean_rtok], [reason_acc], c="r", marker="*", s=90, label="always-reason")
    plt.xlabel("mean generated tokens / question"); plt.ylabel("routed accuracy")
    plt.title("Accuracy vs compute (CALM-R3 frontier)"); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(R3_DIR, "phase0_acc_vs_compute.png"), dpi=200); plt.close()

    out = dict(n=n, T_fast=T, fast_acc=fast_acc, reason_acc=reason_acc, mean_reason_tokens=mean_rtok,
               best=best, reason_helps_bottom30=reason_helps_low, verdict=verdict, buckets=rows, sweep=sweep)
    try:
        with open(os.path.join(R3_DIR, "phase0_results.json"), "w") as f: json.dump(out, f, indent=2)
    except Exception as _e: print("  [warn] save json:", _e)
    print("saved Phase 0 artifacts ->", R3_DIR)
    return out

phase0_out = phase0()


# =====================================================================================



## PHASE 1 — two-adapter routing (gated on GO)


In [ ]:
# PHASE 1 -- TWO-ADAPTER ROUTING + corrected Reflective Correction Loss  (gated)
#   Fast path  = existing trained adapter ("default").
#   Reflect path = new "reflect" LoRA adapter, trained with:
#     L = CE(reflect, y) on HARD items (fast wrong / unsure)               [teach correction]
#       + mu * KL( stopgrad softmax(fast) || softmax(reflect) ) on EASY    [preserve instinct]
#   Reasoning is conditioned on an OFFLINE rationale (no in-loop generation -> budget-safe).
# =====================================================================================
HARD_N, EASY_N = 400, 200
RFT_EPOCHS, RFT_LR, RFT_BS, RFT_ACC, RFT_MU = 2, 1e-4, 2, 8, 0.3
RAT_TOK = 160  # rationale length used for training prompts


def _adapter(name):
    try: student.set_adapter(name)
    except Exception as e: print("  set_adapter(%s) warn: %s" % (name, e))

@torch.no_grad()
def _offline_rationale(q, opts):
    _use_cache(True)
    txt, _, _ = gen_rationale(q, opts, max_new=RAT_TOK)
    _use_cache(False)
    return txt

def _score_prompt(q, opts, rationale):
    return (format_prompt(q, opts) + "\n\nStep-by-step reasoning:\n" + rationale.strip() +
            "\n\nGiven the reasoning above, reply with ONLY one letter: A, B, C, or D.")

def _letter_logits_grad(prompts):
    chats = [stu_chat(p) for p in prompts]
    enc = student_tok(chats, return_tensors="pt", padding=True, truncation=True,
                      max_length=cfg.max_src_len + RAT_TOK + 64).to(STUDENT_DEVICE)
    last = _fwd_last(student, enc.input_ids, enc.attention_mask)   # grad ON
    return _letter_logits(last)

def phase1():
    from peft import LoraConfig
    import random
    print("\n" + "=" * 70 + "\nPHASE 1  two-adapter routing + Reflective Correction Loss\n" + "=" * 70)
    student.eval(); _adapter("default")

    # ---- split seed_ds into EASY (fast right & confident) and HARD (fast wrong / unsure) ----
    T = phase0_out["T_fast"]
    sp = [format_prompt(e["question"], e["options"]) for e in seed_ds]
    slog = student_option_logits(student, sp).numpy(); sP = _sm(slog, T)
    sg = np.array([OPTIONS.index(e["answer"]) for e in seed_ds])
    spred = sP.argmax(1); sconf = sP.max(1); scorrect = spred == sg
    hard_i = [i for i in range(len(seed_ds)) if (not scorrect[i]) or sconf[i] < 0.5][:HARD_N]
    easy_i = [i for i in range(len(seed_ds)) if scorrect[i] and sconf[i] >= 0.7][:EASY_N]
    print("hard=%d easy=%d" % (len(hard_i), len(easy_i)))

    # ---- offline rationales + training examples ----
    _use_cache(True)
    ex = []   # (score_prompt, gold_idx, type, soft_target_or_None)
    for k, i in enumerate(hard_i):
        if not have_time(700): print("  [budget] stop hard-rationale at", k); break
        e = seed_ds[i]; rat = _offline_rationale(e["question"], e["options"])
        ex.append((_score_prompt(e["question"], e["options"], rat), int(sg[i]), "hard", None))
    for k, i in enumerate(easy_i):
        if not have_time(700): print("  [budget] stop easy-rationale at", k); break
        e = seed_ds[i]; rat = _offline_rationale(e["question"], e["options"])
        ex.append((_score_prompt(e["question"], e["options"], rat), int(sg[i]),
                   "easy", sP[i].astype(np.float32)))
    _use_cache(False)
    random.Random(cfg.seed).shuffle(ex)
    print("training examples:", len(ex))

    # ---- add + activate the reflect adapter; train only its params ----
    rcfg = LoraConfig(r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
                      bias="none", task_type="CAUSAL_LM", target_modules=list(cfg.lora_targets))
    try: student.add_adapter("reflect", rcfg)
    except Exception as e: print("  add_adapter warn (may already exist):", e)
    _adapter("reflect")
    for n_, p_ in student.named_parameters():
        p_.requires_grad = ("lora" in n_ and "reflect" in n_)
    params = [p for p in student.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=RFT_LR); _use_cache(False); student.train()
    print("reflect trainable tensors:", len(params))

    steps = 0
    for ep in range(RFT_EPOCHS):
        for b in range(0, len(ex), RFT_BS):
            if not have_time(600): print("  [budget] stop training"); break
            batch = ex[b:b + RFT_BS]
            logits = _letter_logits_grad([z[0] for z in batch]).float()
            gold = torch.tensor([z[1] for z in batch], device=logits.device)
            mask_hard = torch.tensor([z[2] == "hard" for z in batch], device=logits.device)
            loss = torch.tensor(0.0, device=logits.device)
            if mask_hard.any():
                loss = loss + F.cross_entropy(logits[mask_hard], gold[mask_hard])
            me = ~mask_hard
            if me.any():
                soft = torch.tensor(np.stack([z[3] for z in batch if z[2] == "easy"]),
                                    device=logits.device).float()
                logp = F.log_softmax(logits[me], -1)
                loss = loss + RFT_MU * F.kl_div(logp, soft, reduction="batchmean")
            (loss / RFT_ACC).backward(); steps += 1
            if steps % RFT_ACC == 0:
                torch.nn.utils.clip_grad_norm_(params, cfg.grad_clip); opt.step(); opt.zero_grad()
        print("  epoch %d done (loss=%.4f)" % (ep + 1, float(loss)))
    student.eval(); _use_cache(False)
    save_dir = os.path.join(R3_DIR, "reflect_adapter")
    student.save_pretrained(save_dir)  # writes all adapters incl. reflect
    print("saved reflect adapter ->", save_dir)

    # ---- routed evaluation on a test subset ----
    tau = phase0_out["best"]["tau"]
    rng = random.Random(cfg.seed + 1); idx = list(range(len(test_ds))); rng.shuffle(idx)
    idx = idx[:min(REASON_TEST_N, len(test_ds))]; sub = [test_ds[i] for i in idx]
    gold = np.array([OPTIONS.index(e["answer"]) for e in sub])
    _adapter("default")
    flog = student_option_logits(student, [format_prompt(e["question"], e["options"]) for e in sub]).numpy()
    fp = _sm(flog, T); fast_pred = fp.argmax(1); fast_conf = fp.max(1)
    route = fast_conf < tau
    pred = fast_pred.copy(); toks = np.zeros(len(sub))
    _use_cache(True)
    for j, e in enumerate(sub):
        if route[j]:
            if not have_time(500): break
            _adapter("reflect"); rat, ntok, letter = gen_rationale(e["question"], e["options"])
            if letter not in OPTIONS:
                letter = OPTIONS[int(reason_logits(e["question"], e["options"], rat).argmax())]
            pred[j] = OPTIONS.index(letter); toks[j] = ntok; _adapter("default")
    _use_cache(False)
    routed_acc = float((pred == gold).mean()); fast_acc = float((fast_pred == gold).mean())
    res = dict(tau=tau, routed_acc=routed_acc, fast_only_acc=fast_acc,
               frac_reasoned=float(route.mean()), mean_tokens=float(toks.mean()),
               delta_vs_fast=routed_acc - fast_acc)
    with open(os.path.join(R3_DIR, "phase1_routed.json"), "w") as f: json.dump(res, f, indent=2)
    print("ROUTED: acc=%.4f (fast-only %.4f, %+.4f) | %.0f%% reasoned | %.1f tok/q"
          % (routed_acc, fast_acc, routed_acc - fast_acc, 100 * route.mean(), toks.mean()))
    return res

if RUN_PHASE1 and phase0_out["verdict"] == "GO":
    phase1_out = phase1()
else:
    print("\nPhase 1 skipped (RUN_PHASE1=%s, verdict=%s)." % (RUN_PHASE1, phase0_out["verdict"]))


# =====================================================================================



## Zip artifacts


In [ ]:
# ZIP all artifacts
# =====================================================================================
import shutil
zip_base = os.path.join(cfg.out_dir, "calm_r3_artifacts")
shutil.make_archive(zip_base, "zip", R3_DIR)
print("\nzipped ->", zip_base + ".zip")
print("contents:", sorted(os.listdir(R3_DIR)))

